In [ ]:
!pip install -r requirements.txt

In [ ]:
import os
from data_generator import collect_all_data
from scraper import scrape_data_to_json
from data_ingestion_v2 import data_loading
from pathlib import Path
import json
import os
import gc
from tqdm import tqdm
from RAG import get_response 

DB_PATH = Path("./corpus_data")

INPUT_FILE = "queries.json"
OUTPUT_FILE = "formatted_outcome.json"


### Running the Inference pipeline

- The below function assumes that the dir contains a file `queries.json`
- The format to the json is supposed to be:<br>
    ``` json 
    {
        "queries": [
            {
              "query_id": "0",
              "query": "How are cream cheese wontons shaped before frying?"
            },
            {
              "query_id": "1",
              "query": "What seasonings were not used in traditional Ainu cuisine but appeared in modern Ainu cuisine?"
            }, .... 
          ]
    }`
- Once done the outputs are created in the formatted_outcome.json

In [ ]:
## Inference - Make sure to add a file quries.json

def main():
    if not os.path.exists(INPUT_FILE):
        print(f"{INPUT_FILE} not found!")
        return

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        input_data = json.load(f)

    queries = input_data.get("queries", [])
    results = []

    print(f"Running RAG on {len(queries)} test queries...")

    for item in tqdm(queries):
        query_id = item.get("query_id")
        query_text = item.get("query")

        try:
            response_text, sources, top_docs = get_response(query_text)

            formatted_context = []
            for i, doc in enumerate(top_docs):
                formatted_context.append({
                    "doc_id": f"{i:03d}",
                    "text": doc.page_content.strip()
                })

            results.append({
                "query_id": query_id,
                "query": query_text,
                "response": response_text,
                "retrieved_context": formatted_context
            })
            
            gc.collect()

        except Exception as e:
            print(f"\nError on query_id {query_id}: {e}")
            results.append({
                "query_id": query_id,
                "query": query_text,
                "response": "I dont know the answer (Error occurred)",
                "retrieved_context": []
            })
            gc.collect()
            continue

    # 4. Final output structure
    output_data = {"results": results}

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=2, ensure_ascii=False)

    print(f"\nResults saved to {OUTPUT_FILE}")


main()

### Full Pipeline:

- While this may take some time as contains all the parts from data collection to vectorization one have toensure a good internet connection, avoid firewalls or allow the wiki pages and required urls in the firewall.<br>

- `Delete` the `./Vector_Storage_MasterChef` and `./corpus_data`

- Run the below code.

- This may throw error if the filepaths mismatch for `Windows` or `Linux` as the code is tested on `MAC`

- Once the full data is created check if `105 documents` are processed if not wait for some time and try again. As you may be blocked by wiki bot identifier.


In [ ]:
## Run the Context Creation.


def run_pipeline():
    print("--- Phase 1: Collecting Sources ---")
    collect_all_data()
    
    print("--- Phase 2: Scraping Data ---")
    scrape_data_to_json()
    
    print("--- Phase 3: Ingesting into Chroma ---")
    data_loading(str(DB_PATH))


run_pipeline()